# Pseudo Label（疑似ラベリング）csvファイル生成コード
- 学習済みSEDモデルを使って、ラベルなしデータに「予測ラベル」を自動付与し、疑似ラベルCSVを生成する。
- 付与したラベルを正解として再学習することで、モデルの精度を向上させる。

## 前提条件
- Kaggle Notebookのオンライン環境で動作させる。
- 学習済みSEDモデルが存在する。

## 変更履歴
- 変更009
  - OOF分離: fold kで学習したモデルはfold kのデータを推論しない（リーク防止）
  - fold分割: Pseudo label自体をStratifiedGroupKFoldで5分割
  - フィルタリング: primary_label_prob > 0.5
  - 出力: submission_sed_full.csv（全件）+ submission_sed_full_filtered.csv（フィルタ済み・fold_id付き）


In [ ]:
# ================================================================
# S1 -- 設定・ライブラリ
# ================================================================
import os, re, gc
import numpy as np
import pandas as pd
import librosa  # 音声処理ライブラリ
import soundfile as sf  # 音声ファイルの読み込みライブラリ
import onnxruntime as ort  # 学習済みモデルの高速実行用ライブラリ
from pathlib import Path
from scipy.ndimage import gaussian_filter1d  # 予測値を滑らかにするフィルタ
from sklearn.model_selection import StratifiedGroupKFold  # データ分割ライブラリ

# --- パス設定 ---
# 各種ファイルの保存先・読み込み元のパスを設定する
BASE            = Path("/kaggle/input/competitions/birdclef-2026")
SED_OUTPUT_PATH = Path("/kaggle/working/submission_sed_full.csv")  # 全件の推論結果
FILTERED_OUTPUT = Path("/kaggle/working/submission_sed_full_filtered.csv")  # 疑似ラベルCSV
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

# --- 音声設定 ---
SR             = 32000            # サンプリングレート（1秒あたりのデータ点数）
WINDOW_SEC     = 5                # 1回の推論で処理する音声の長さ（秒）
N_WINDOWS      = 12               # 1分間（60秒）を5秒ずつ分割すると12ウィンドウ
WINDOW_SAMPLES = SR * WINDOW_SEC  # 1ウィンドウあたりのサンプル数

# --- SED メルスペクトログラム設定 ---
# 音声をメルスペクトログラム（音声を画像化したもの）に変換する際のパラメータ
N_MELS_SED = 256    # 周波数方向の解像度（縦軸のピクセル数に相当）
N_FFT_SED  = 2048   # フーリエ変換のウィンドウサイズ（大きいほど周波数解像度が高い）
HOP_SED    = 512    # フーリエ変換のステップ幅（小さいほど時間解像度が高い）
FMIN_SED   = 20     # 分析する最低周波数（Hz）
FMAX_SED   = 16000  # 分析する最高周波数（Hz）
TOP_DB_SED = 80     # デシベルの上限（これ以上小さい音は切り捨て）

# --- フィルタ設定 ---
# 疑似ラベルの品質を保つため、予測確率が低いものを除外する
PRIMARY_LABEL_PROB_THR = 0.5  # 最大予測確率がこの値以上のものだけを疑似ラベルとして採用
N_FOLDS                = 5    # 交差検証の分割数

# --- クラスリスト ---
# sample_submission.csvから予測対象の234種のラベルリストを取得する
sample_sub    = pd.read_csv(BASE / "sample_submission.csv")
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
print(f"Classes: {N_CLASSES}")

# --- 推論対象ファイルの取得 ---
# 本番提出時はtest_soundscapes、ドライラン時はtrain_soundscapesを使用する
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    print("Hidden test not mounted. Using train_soundscapes (all files).")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))
print(f"Target files: {len(test_paths)}")

In [ ]:
# ================================================================
# S2 -- SED推論ヘルパー関数
# ================================================================
# 音声ファイルをSEDモデルで推論するための関数を定義する。
# ================================================================

def find_sed_dir():
    """
    学習済みSEDモデル（ONNX形式）のディレクトリを自動検索して返す。
    sed_fold0.onnxが存在するディレクトリを探す。
    """
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError("sed_fold0.onnx not found.")
    return hits[0].parent

def make_sed_session(path, providers=["CPUExecutionProvider"]):
    """
    ONNXモデルの推論セッションを作成する。
    intra_op_num_threads=4: 並列処理スレッド数
    ORT_ENABLE_ALL: グラフ最適化を最大限有効化して高速化
    """
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=providers)

def audio_to_mel(chunks):
    """
    音声チャンク（波形データ）をメルスペクトログラムに変換する。
    メルスペクトログラム = 音声を画像化したもの（縦軸:周波数、横軸:時間）
    正規化（平均0・標準偏差1）により、音量の違いによる影響を除去する。
    """
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
                                            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)  # デシベルスケールに変換
        s = (s - s.mean()) / (s.std() + 1e-6)          # 正規化（平均0・標準偏差1）
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)  # (N, 1, mel, time)の形状に変換

def file_to_sed_chunks(path):
    """
    1分間の音声ファイルを5秒×12チャンクに分割する。
    サンプリングレートが異なる場合はリサンプリングする。
    60秒に満たない場合はゼロパディングで補完する。
    """
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)               # ステレオ→モノラル変換
    if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)  # リサンプリング
    n = 60 * SR
    if len(y) < n: y = np.pad(y, (0, n - len(y)))    # 60秒に満たない場合はゼロパディング
    else:          y = y[:n]                           # 60秒を超える場合はカット
    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)     # (12, 160000)に分割
    ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC # [5, 10, 15, ..., 60]
    return chunks, ends

def sigmoid_sed(x):
    """
    ロジット（生の予測値）を確率（0〜1）に変換するシグモイド関数。
    オーバーフロー防止のため-50〜50にクリップする。
    """
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

def infer_files(file_list, sessions):
    """
    指定ファイルリストをsessionsで推論してDataFrameを返す。
    clip予測とframe_max予測を0.5ずつブレンドする（推論コードと同じ方式）。
    ガウシアンフィルタで時間方向に平滑化する（隣接ウィンドウの予測を少し混ぜる）。
    """
    rows, preds = [], []
    for path in file_list:
        chunks, ends = file_to_sed_chunks(path)
        mel = audio_to_mel(chunks)
        p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)
        for sess in sessions:
            outs = sess.run(None, {sess.get_inputs()[0].name: mel})
            # clip_logits（クリップ全体の予測）とframe_max（フレームの最大値）を等重みでブレンド
            p_sum += 0.5 * sigmoid_sed(outs[0]) + 0.5 * sigmoid_sed(outs[1].max(axis=1))
        p_mean = p_sum / len(sessions)  # 全foldの平均を取る
        if len(p_mean) > 1:
            # 隣接ウィンドウの予測を少し混ぜて滑らかにする（sigma=0.65）
            p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
        stem = Path(path).stem
        rows.extend([f"{stem}_{int(t)}" for t in ends])
        preds.append(p_mean)
    arr = np.concatenate(preds, axis=0)
    df = pd.DataFrame(np.clip(arr, 0.0, 1.0), columns=PRIMARY_LABELS)
    df.insert(0, "row_id", rows)
    return df

# SED学習済みモデルの読み込み
sed_dir = find_sed_dir()
sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                         key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
print(f"SED dir: {sed_dir}")
print(f"SED folds: {[p.name for p in sed_fold_paths]}")

In [ ]:
# ================================================================
# S3 -- OOF分離推論（リーク防止）
# ================================================================
# 「OOF（Out-Of-Fold）」とは交差検証で使われる手法。
# fold kで学習したモデルはfold kのデータを学習に使っているため、
# そのデータを推論すると「カンニング」になってしまう（データリーク）。
# これを防ぐため、fold kのファイルはfold k以外のモデルで推論する。
#
# 例: fold 0のファイル → fold 1,2,3,4のモデルで推論して平均を取る
# ================================================================

# ファイルをfoldに振り分け（ファイル番号を5で割った余りをfold番号とする）
n_files = len(test_paths)
file_fold_ids = np.arange(n_files) % N_FOLDS  # 0,1,2,3,4,0,1,2,3,4,...と順番に割り当て

print(f"Total files: {n_files}")
for f in range(N_FOLDS):
    print(f"  fold{f}: {(file_fold_ids == f).sum()} files")

all_dfs = []

for fold_k in range(N_FOLDS):
    cache_path = Path(f"/kaggle/working/fold{fold_k}_cache.csv")

    # --- キャッシュがあれば再計算をスキップ ---
    # 途中でセッションが切れても最初からやり直さなくて済むようにキャッシュを使う
    if cache_path.exists():
        print(f"fold{fold_k}: キャッシュ読み込み ({cache_path})")
        df_k = pd.read_csv(cache_path)
        all_dfs.append(df_k)
        print(f"  fold{fold_k} loaded: {df_k.shape}")
        continue

    # fold_kのファイルを推論するモデル: fold_k以外の全モデル（リーク防止）
    target_files = [p for p, fid in zip(test_paths, file_fold_ids) if fid == fold_k]
    other_fold_paths = [p for i, p in enumerate(sed_fold_paths) if i != fold_k]
    sessions = [make_sed_session(p) for p in other_fold_paths]

    print(f"fold{fold_k}: {len(target_files)} files, using {len(sessions)} models")
    df_k = infer_files(target_files, sessions)

    # キャッシュ保存（途中で止まっても再実行時にスキップできる）
    df_k.to_csv(cache_path, index=False)
    print(f"  fold{fold_k} done & cached: {df_k.shape}")

    all_dfs.append(df_k)
    del sessions; gc.collect()

# 全foldの結果を結合して1つのCSVに保存
sed_sub = pd.concat(all_dfs, ignore_index=True)
sed_sub.to_csv(SED_OUTPUT_PATH, index=False)
print(f"\nsubmission_sed_full.csv saved. Shape: {sed_sub.shape}")

In [ ]:
# ================================================================
# S4 -- フィルタリング・fold分割
# ================================================================
# 疑似ラベルの品質管理を行う。
# 予測確率が低いもの（モデルが自信を持っていないもの）は
# ノイズになる可能性が高いため除外する。
# ================================================================

# 各ウィンドウの「最も確率が高いクラス」とその確率を計算する
# score_matrix: (全ウィンドウ数, 234クラス)の確率行列
score_matrix = sed_sub[PRIMARY_LABELS].values
primary_label_idx  = score_matrix.argmax(axis=1)   # 最大確率のクラスのインデックス
primary_label_prob = score_matrix.max(axis=1)       # 最大確率の値
primary_label      = [PRIMARY_LABELS[i] for i in primary_label_idx]  # クラス名

sed_sub["primary_label"]      = primary_label       # 最も確率が高い種のラベル
sed_sub["primary_label_prob"] = primary_label_prob  # その確率
sed_sub["sample_id"]          = sed_sub["row_id"].apply(lambda x: "_".join(x.split("_")[:-1]))

print(f"Before filter: {len(sed_sub)}")
# 最大確率が閾値（0.5）以上のウィンドウのみを疑似ラベルとして採用
filtered = sed_sub[sed_sub["primary_label_prob"] > PRIMARY_LABEL_PROB_THR].reset_index(drop=True)
print(f"After filter (prob > {PRIMARY_LABEL_PROB_THR}): {len(filtered)}")

# StratifiedGroupKFoldで5分割する
# Stratified: 各foldで種のバランスを保つ
# Group: 同じ音声ファイルのウィンドウが同じfoldに入るようにする（リーク防止）
cv_split = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42).split(
    filtered,
    filtered["primary_label"],  # 種のバランスを保つための層化基準
    filtered["sample_id"]        # 同じファイルは同じfoldに入れるためのグループ
))

# fold_idを付与（後続の学習コードでどのfoldか識別するために使う）
filtered["fold_id"] = None
for fold_id, (train_fold, val_fold) in enumerate(cv_split):
    filtered.loc[filtered.index[val_fold], "fold_id"] = fold_id

print(f"fold_id分布:")
print(filtered["fold_id"].value_counts().sort_index())

# フィルタ済み・fold_id付きのCSVを保存（これが次の学習ステップで使う疑似ラベル）
filtered.drop(columns=["sample_id"]).to_csv(FILTERED_OUTPUT, index=False)
print(f"\nsubmission_sed_full_filtered.csv saved. Shape: {filtered.shape}")

In [ ]:
# ================================================================
# S5 -- submission.csv生成
# ================================================================
# Kaggleへの提出用ファイルを生成する。
# Pseudo Label（疑似ラベリング）CSVファイルとは無関係。
# ドライラン時（本番データなし）はtrain_soundscapesで代用する。
# ================================================================
if IS_DRY_RUN:
    # ドライラン: sample_submissionの行に推論結果を埋め込む
    sub_out = sample_sub.copy()
    sed_indexed = sed_sub.set_index("row_id")
    for col in PRIMARY_LABELS:
        sub_out[col] = sub_out["row_id"].map(sed_indexed[col]).fillna(0.5).astype(np.float32)
else:
    # 本番: 推論結果をそのまま提出用CSVとして保存
    sub_out = sed_sub[["row_id"] + PRIMARY_LABELS].copy()

sub_out.to_csv(SUBMISSION_PATH, index=False)
print(f"submission.csv saved. Shape: {sub_out.shape}")
print(f"Score range: {sub_out[PRIMARY_LABELS].min().min():.4f} to {sub_out[PRIMARY_LABELS].max().max():.4f}")

In [ ]:
# ================================================================
# S5 -- submission.csv生成（Kaggle用の処理。Pseudo Label（疑似ラベリング）csvファイルとは無関係。）
# ================================================================
if IS_DRY_RUN:
    sub_out = sample_sub.copy()
    sed_indexed = sed_sub.set_index("row_id")
    for col in PRIMARY_LABELS:
        sub_out[col] = sub_out["row_id"].map(sed_indexed[col]).fillna(0.5).astype(np.float32)
else:
    sub_out = sed_sub[['row_id'] + PRIMARY_LABELS].copy()

sub_out.to_csv(SUBMISSION_PATH, index=False)
print(f"submission.csv saved. Shape: {sub_out.shape}")
print(f"Score range: {sub_out[PRIMARY_LABELS].min().min():.4f} to {sub_out[PRIMARY_LABELS].max().max():.4f}")